In [0]:
%run ./00_config

In [0]:
%sql
-- Create a share for all tables in the nyc_rats catalog
-- The recipient will ONLY be able to access tables added to this share
CREATE SHARE IF NOT EXISTS hackathon_share;

In [0]:
# Add tables to the share (idempotent — skips tables already in the share)
tables_to_share = [
    "nyc_rats.01_bronze.rat_sightings_bronze",
    "nyc_rats.01_bronze.restaurant_inspections_bronze",
    "nyc_rats.02_silver.rodent_complaints_silver",
    "nyc_rats.02_silver.restaurant_violations_silver",
    "nyc_rats.02_silver.restaurants_silver",
    "nyc_rats.03_gold.zip_service_gap_gold",
    "nyc_rats.03_gold.top_rodent_cuisines_gold",
    "nyc_rats.03_gold.zip_monthly_trend_gold",
    "nyc_rats.03_gold.rodent_rate_by_boro_gold"
]

existing_rows = spark.sql("SHOW ALL IN SHARE hackathon_share").collect()
existing_tables = {r["shared_object"] for r in existing_rows}

for table in tables_to_share:
    if table in existing_tables:
        print(f"  ✓ {table} (already shared)")
    else:
        spark.sql(f"ALTER SHARE hackathon_share ADD TABLE {table}")
        print(f"  + {table} (newly added)")

In [0]:
%sql
-- Verify tables exist in the share
SHOW ALL IN SHARE hackathon_share;

In [0]:

# Register recipient using their catalog sharing ID stored as secret
# Using Databricks CLI: databricks secrets put-secret hackathon mitchell_sharing_id --string-value '<sharing_id>'
_mate_sharing_id = dbutils.secrets.get(scope='hackathon', key='mitchell_sharing_id')
spark.sql(f"CREATE RECIPIENT IF NOT EXISTS hackathon_mate_mitchell USING ID '{_mate_sharing_id}'")

In [0]:
%sql
-- Grant read access to the share
GRANT SELECT ON SHARE hackathon_share TO RECIPIENT hackathon_mate_mitchell;

In [0]:
%sql
-- Verify: show all tables in the share and the recipient's grants
SHOW ALL IN SHARE hackathon_share;
SHOW GRANTS ON SHARE hackathon_share;

In [0]:
%sql
DESCRIBE RECIPIENT hackathon_mate_mitchell